<a href="https://colab.research.google.com/github/vtbranco07-del/vt_python/blob/main/Aula011_Vitor_branco_Atividade.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
!pip install openpyxl

In [3]:

# ============================================================
# ATIVIDADE AULA 11
# EXTRATOR DE DADOS FISCAIS: XML PARA EXCEL
# ============================================================

# ETAPA 1 - IMPORTAÇÕES E PADRÃO PEP 8

import xml.etree.ElementTree as ET
import csv
import logging
import io

from decimal import Decimal
from google.colab import files

from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter


# ============================================================
# CONFIGURAÇÕES INICIAIS
# ============================================================

NAMESPACE = {
    "nfe": "http://www.portalfiscal.inf.br/nfe"
}

COLUNAS = [
    "Número da Nota",
    "Data de Emissão",
    "Nome do Fornecedor / Emitente",
    "CNPJ do Emitente",
    "Valor Total da Nota"
]

logging.basicConfig(
    level=logging.INFO,
    format="%(levelname)s: %(message)s"
)


# ============================================================
# ETAPA 2 - LEITURA E VARREDURA DOS XMLs
# ============================================================

def obter_texto(elemento, caminho, padrao="Não informado"):
    """
    Localiza um elemento no XML e retorna seu conteúdo.
    Caso não encontre, retorna um valor padrão.
    """

    resultado = elemento.find(caminho, NAMESPACE)

    if resultado is not None and resultado.text:
        return resultado.text.strip()

    return padrao


# ============================================================
# ETAPA 3 - EXTRAÇÃO DOS DADOS FISCAIS
# ============================================================

def extrair_dados_nfe(arquivo):
    """
    Realiza a leitura do XML e extrai:
    - Número da nota;
    - Data de emissão;
    - Nome do emitente;
    - CNPJ do emitente;
    - Valor total da nota.
    """

    arvore = ET.parse(arquivo)
    raiz = arvore.getroot()

    # Identifica a estrutura principal do XML.
    if raiz.tag.endswith("nfeProc"):
        no_nfe = raiz.find("nfe:NFe", NAMESPACE)

    elif raiz.tag.endswith("NFe"):
        no_nfe = raiz

    else:
        no_nfe = raiz.find(".//nfe:NFe", NAMESPACE)

    if no_nfe is None:
        raise ValueError("Estrutura de NF-e não encontrada.")

    # Localiza as informações principais da nota.
    inf_nfe = no_nfe.find("nfe:infNFe", NAMESPACE)

    if inf_nfe is None:
        raise ValueError("Informações da NF-e não encontradas.")

    # Extrai o número da nota.
    numero = obter_texto(
        inf_nfe,
        "nfe:ide/nfe:nNF"
    )

    # Extrai a data de emissão.
    data = obter_texto(
        inf_nfe,
        "nfe:ide/nfe:dhEmi",
        obter_texto(
            inf_nfe,
            "nfe:ide/nfe:dEmi"
        )
    )

    # Localiza os dados do emitente.
    emitente = inf_nfe.find(
        "nfe:emit",
        NAMESPACE
    )

    if emitente is None:
        raise ValueError("Dados do emitente não encontrados.")

    # Extrai o nome do fornecedor.
    nome = obter_texto(
        emitente,
        "nfe:xNome"
    )

    # Extrai o CNPJ.
    cnpj = obter_texto(
        emitente,
        "nfe:CNPJ"
    )

    # Extrai o valor total da nota fiscal.
    valor = obter_texto(
        inf_nfe,
        "nfe:total/nfe:ICMSTot/nfe:vNF",
        "0.00"
    )

    valor = Decimal(valor.replace(",", "."))

    # Retorna todos os dados extraídos.
    return {
        "Número da Nota": numero,
        "Data de Emissão": data,
        "Nome do Fornecedor / Emitente": nome,
        "CNPJ do Emitente": cnpj,
        "Valor Total da Nota": valor
    }


# ============================================================
# VARREDURA E PROCESSAMENTO DOS ARQUIVOS
# ============================================================

def percorrer_xmls(arquivos):
    """
    Percorre os arquivos enviados, realizando a leitura
    individual de cada XML e armazenando os dados extraídos.
    """

    registros = []

    for nome_arquivo, conteudo in arquivos.items():

        if not nome_arquivo.lower().endswith(".xml"):
            logging.warning(
                "Arquivo ignorado: %s",
                nome_arquivo
            )
            continue

        try:
            arquivo = io.BytesIO(conteudo)

            dados = extrair_dados_nfe(arquivo)

            registros.append(dados)

            logging.info(
                "Arquivo processado: %s",
                nome_arquivo
            )

        except (
            ET.ParseError,
            OSError,
            ValueError,
            TypeError
        ) as erro:

            logging.error(
                "Erro ao processar %s: %s",
                nome_arquivo,
                erro
            )

    return registros


# ============================================================
# ETAPA 4 - GERAÇÃO DO ARQUIVO CSV
# ============================================================

def gerar_csv(registros, caminho):
    """
    Cria um arquivo CSV contendo os dados
    extraídos de todas as notas fiscais.
    """

    with open(
        caminho,
        "w",
        newline="",
        encoding="utf-8-sig"
    ) as arquivo:

        escritor = csv.DictWriter(
            arquivo,
            fieldnames=COLUNAS,
            delimiter=";"
        )

        escritor.writeheader()

        for registro in registros:

            linha = registro.copy()

            linha["Valor Total da Nota"] = (
                f"{registro['Valor Total da Nota']:.2f}"
            )

            escritor.writerow(linha)


# ============================================================
# GERAÇÃO DA PLANILHA EXCEL
# ============================================================

def gerar_excel(registros, caminho):
    """
    Cria uma planilha Excel com os dados extraídos,
    aplicando formatação ao cabeçalho e aos valores monetários.
    """

    planilha = Workbook()

    aba = planilha.active
    aba.title = "Notas Fiscais"

    # Insere o cabeçalho.
    aba.append(COLUNAS)

    # Insere os dados das notas fiscais.
    for registro in registros:

        linha = [
            registro[coluna]
            for coluna in COLUNAS
        ]

        aba.append(linha)

    # Formatação do cabeçalho.
    for celula in aba[1]:

        celula.font = Font(
            bold=True,
            color="FFFFFF"
        )

        celula.fill = PatternFill(
            fill_type="solid",
            fgColor="17365D"
        )

        celula.alignment = Alignment(
            horizontal="center",
            vertical="center"
        )

    # Formatação dos valores monetários.
    for linha in range(2, aba.max_row + 1):

        aba.cell(
            row=linha,
            column=5
        ).number_format = '"R$" #,##0.00'

    # Ajusta a largura das colunas.
    larguras = [18, 25, 42, 22, 25]

    for indice, largura in enumerate(larguras, start=1):

        coluna = get_column_letter(indice)

        aba.column_dimensions[coluna].width = largura

    # Melhorias de visualização.
    aba.freeze_panes = "A2"
    aba.auto_filter.ref = aba.dimensions

    # Salva o arquivo Excel.
    planilha.save(caminho)


# ============================================================
# FUNÇÃO PRINCIPAL
# ============================================================

def main():
    """
    Executa todas as etapas do programa:
    1. Recebe os arquivos XML;
    2. Percorre e processa os arquivos;
    3. Extrai os dados fiscais;
    4. Gera os arquivos Excel e CSV;
    5. Disponibiliza os arquivos para download.
    """

    print("=" * 55)
    print("EXTRATOR DE DADOS FISCAIS - XML PARA EXCEL")
    print("=" * 55)

    # Solicita o envio dos arquivos XML.
    print("\nSelecione os arquivos XML das notas fiscais:")

    arquivos = files.upload()

    if not arquivos:
        print("\nNenhum arquivo foi enviado.")
        return

    # Realiza a leitura e extração.
    print("\nIniciando processamento dos arquivos...\n")

    registros = percorrer_xmls(arquivos)

    if not registros:
        print("\nNenhuma nota fiscal válida foi encontrada.")
        return

    # Gera os arquivos de saída.
    gerar_csv(
        registros,
        "notas_fiscais.csv"
    )

    gerar_excel(
        registros,
        "notas_fiscais.xlsx"
    )

    # Exibe o resultado do processamento.
    print("\n" + "=" * 55)
    print("PROCESSAMENTO CONCLUÍDO COM SUCESSO!")
    print("=" * 55)

    print(f"\nTotal de notas extraídas: {len(registros)}")

    print("\nArquivos gerados:")
    print("1. notas_fiscais.xlsx")
    print("2. notas_fiscais.csv")

    # Disponibiliza os arquivos para download.
    print("\nIniciando downloads...\n")

    files.download("notas_fiscais.xlsx")
    files.download("notas_fiscais.csv")


# ============================================================
# INICIALIZAÇÃO DO PROGRAMA
# ============================================================

if __name__ == "__main__":
    main()


EXTRATOR DE DADOS FISCAIS - XML PARA EXCEL

Selecione os arquivos XML das notas fiscais:


Saving nota_fiscal_001.xml to nota_fiscal_001.xml
Saving nota_fiscal_002.xml to nota_fiscal_002.xml
Saving nota_fiscal_003.xml to nota_fiscal_003.xml

Iniciando processamento dos arquivos...


PROCESSAMENTO CONCLUÍDO COM SUCESSO!

Total de notas extraídas: 3

Arquivos gerados:
1. notas_fiscais.xlsx
2. notas_fiscais.csv

Iniciando downloads...



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>